In [ ]:
## Import Jim-relatred stuff

import os
os.environ['JAX_PLATFORMS'] = 'cpu'
import numpy as np
from functools import partial

print("Importing JAX")
import jax
import jax.numpy as jnp
print("Importing JAX successful")

import jimgw
print(jimgw.__file__)
from jimgw.core.single_event.data import Data
jax.config.update("jax_enable_x64", True)

Importing JAX
Importing JAX successful
/users/hin-wai.leong/src/jax-gw-team/jim/src/jimgw/__init__.py


In [10]:
frequencies = jnp.arange(20.0, 512.0, 1/16)
frequencies = jnp.linspace(20.0, 512.0, 7873)
strain_data = frequencies ** 2 + 1j * frequencies ** 3

Data.from_fd(
    frequencies=frequencies,
    fd=strain_data,
    name='TestData'
)

AssertionError: Frequencies do not match after slicing

In [13]:
jnp.diff(frequencies) == (frequencies[1] - frequencies[0])

Array([ True,  True,  True, ...,  True,  True, False], dtype=bool)

In [11]:
jnp.diff(frequencies) - (frequencies[1] - frequencies[0])

Array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        0.00000000e+00,  0.00000000e+00, -5.68434189e-14],      dtype=float64)

In [15]:
frequencies = jnp.linspace(20.0, 512.0, 7800)
df = jnp.diff(frequencies)
jnp.all(df == df[0])

Array(False, dtype=bool)

In [ ]:
print(
    jnp.all(df == df[0]),
    jnp.mean(jnp.abs(df - df[0]))
)

False 1.8102486072811016e-14


In [ ]:
def from_fd(
        fd_strain,
        frequencies,
        epoch: float = 0.0,
        name: str = "",
):
        """Create a Data object starting from (potentially incomplete)
        Fourier domain data.

        Args:
            fd: Fourier domain data array.
            frequencies: Frequencies of the data in Hz.
            epoch: Epoch of the data in seconds (default: 0).
            name: Name of the data (default: '').

        Returns:
            Data: Data object with the Fourier and time domain data.
        """
        delta_f = frequencies[1] - frequencies[0]
        f_nyq = frequencies[-1]
        n_samples = int(np.round(2 * f_nyq / delta_f))
        # Ensure time-domain samples will be even
        if (n_samples % 2) != 0:
            raise ValueError(
                "The number of time-domain samples will not be even. " + 
                "Please check your frequency array."
            )

        # Construct the full frequency array
        n_frequencies = int(np.round(n_samples / 2) + 1)
        freqs = jnp.arange(n_frequencies) * delta_f
        # Fill in the full data array
        start_idx = jnp.searchsorted(freqs, frequencies[0])
        data_fd_full = jax.lax.dynamic_update_slice(
            jnp.zeros_like(freqs, dtype=fd_strain.dtype), fd_strain, (start_idx,))
        # IFFT into time domain
        delta_t = 1 / (2 * f_nyq)
        data_td_full = jnp.fft.irfft(data_fd_full) / delta_t
        # Check frequencies
        assert jnp.array_equal(
            freqs, jnp.fft.rfftfreq(len(data_td_full), delta_t)
        ), "Generated frequencies do not match the input frequencies"
        # Create a Data object
        data = Data(data_td_full, delta_t, epoch=epoch, name=name)
        data.fd = data_fd_full
        # This ensures the newly constructed Data in FD faithfully
        # represents the input FD data.
        d_new, f_new = data.frequency_slice(frequencies[0], frequencies[-1])
        assert jnp.array_equal(d_new, fd_strain), "Data do not match after slicing"
        assert jnp.array_equal(f_new, frequencies), "Frequencies do not match after slicing"
        return data

In [94]:
frequencies = jnp.arange(20.0  + 14/32, 512.5 + 13/32, 1/16)
print(frequencies[-1])
strain_data = frequencies ** 2 + 1j * frequencies ** 3
from_fd(
    frequencies=frequencies,
    fd=strain_data,
    name='TestData'
)

512.875
16412 0
16412 8207 8207 16412


Data(name='TestData', delta_t=0.0009748964172556666, epoch=0.0)

In [85]:
8207 % 2

1

In [83]:
np.round((512.5 + 13/32) * 32)

16413.0

In [76]:
2 * 16 * (128 + 15 / 32)

4111.0

In [36]:
(511.0 + 1/16) % 2

1.0625

In [37]:
(511.0 - 1/16) * 16

8175.0

In [43]:
sliced_data = strain_data[100:]

## Little Performance Test

In [44]:
%timeit jax.lax.dynamic_update_slice(jnp.zeros_like(frequencies, dtype=strain_data.dtype), sliced_data, (100,))

377 μs ± 489 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [45]:
%timeit jnp.zeros_like(frequencies, dtype=strain_data.dtype).at[100:].set(sliced_data)

987 μs ± 12.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Test `arr.at[slice].set(...)` with `jit`

In [46]:
@jax.jit
def test_dynamic_update_slice(frequencies, strain_data):
    start_idx = jnp.searchsorted(frequencies, 20.)
    zeros = jnp.zeros_like(frequencies, dtype=strain_data.dtype)
    return zeros.at[start_idx:].set(strain_data)

In [47]:
test_dynamic_update_slice(frequencies, strain_data[frequencies >= 20.0])

IndexError: Array slice indices must have static start/stop/step to be used with NumPy indexing syntax. Found slice(Traced<ShapedArray(int32[])>with<DynamicJaxprTrace>, None, None). To index a statically sized array at a dynamic position, try lax.dynamic_slice/dynamic_update_slice (JAX does not support dynamically sized arrays within JIT compiled functions).

In [ ]:
f_max = 512.0
duration = 16.0
sampling_freq = 512.0 * 2.0
number_of_samples = int(duration * sampling_freq)
number_of_frequencies = int(number_of_samples / 2 + 1)

frequencies = jnp.linspace(20.0, f_max, 1/16)
print(number_of_frequencies)